# Gift Concierge Agent — End-to-End Test

Tests the full pipeline:
- Intent routing (Groq)
- Catalog search (Qdrant RAG + Groq)
- Logistics check (rule-based + Groq narration)
- Preference update (Groq extraction + Supabase)
- Chitchat (Groq)

In [1]:
import sys
sys.path.insert(0, '../src')

In [4]:
%pip install groq

from agents.orchestrator import GiftOrchestrator
from memory.st_store import SupabaseSTStore
from memory.profile_store import SupabaseProfileStore
from memory.rag_store import QdrantRAGStore
from memory.embedder import OpenRouterEmbedder

orch = GiftOrchestrator(
    st_store=SupabaseSTStore(),
    profile_store=SupabaseProfileStore(),
    rag_store=QdrantRAGStore(embedder=OpenRouterEmbedder()),
)
print('Orchestrator ready')

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\viraj\Zuu\Gift_Concierge_Agent\.venv\Scripts\python.exe -m pip install --upgrade pip' command.
2026-05-10 16:38:25.531 | INFO     | infastructure.db.supabase_client:get_supabase_client:46 - Supabase client initialised (https://rcxpatxxadqnxubzalxe.supabase.co)
2026-05-10 16:38:25.570 | INFO     | memory.rag_store:__init__:117 - QdrantRAGStore initialised — collection: 'kapruka_agent'


Orchestrator ready


## Helper — pretty print response

In [5]:
def show(r):
    print(f'Intent     : {r.intent} ({r.confidence:.0%})')
    print(f'Session    : {r.session_id}')
    print(f'Metadata   : {r.metadata}')
    print()
    print('Reply:')
    print(r.reply)

## Test 1 — Search (catalog + RAG)

In [7]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Find a dinasoree toy for my kid',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 16:39:52.861 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 11:09:53 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 16:39:53.335 | INFO     | agents.router:classify:125 - Intent: search (98%) — Asking for a specific product recommendation f

Intent     : search (98%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Find a dinasoree toy for my kid', 'recipient': None}

Reply:
I'd be happy to help you find a lovely gift for your kid. Unfortunately, I couldn't find any dinosaur-themed toys in our catalog. However, I can suggest some adorable soft toys that your kid might enjoy.

My top recommendation would be the Peppa Pig Soft Plush Blue 12" (LKR 1,150, https://www.kapruka.com/buyonline/peppa-pig-soft-plush-blue-12-i/kid/softtoy00982). This cute plush toy is a great alternative to a dinosaur toy, and its affordable price makes it an excellent value for money.

If you'd like to consider other options, I can also suggest the SpongeBob SquarePants Plush Soft Toy (LKR 2,200, https://www.kapruka.com/buyonline/spongebob-squarepants-plush-so/kid/softtoy001192) or the Squirtle Pokemon Plush Toy (LKR 2,800, https://www.kapruka.com/buyonline/squirtle-pokemon-plush-toy/kid/softtoy00979). Both of these toys ar

## Test 2 — Preference update

In [8]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='My wife loves dark chocolate and orchids, budget around LKR 5000',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 16:43:21.629 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 11:13:22 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 16:43:22.158 | INFO     | agents.router:classify:125 - Intent: preference_update (96%) — Sharing recipient preferences and b

Intent     : preference_update (96%)
Session    : s001
Metadata   : {'recipient': 'Wife', 'action': 'created a new profile', 'is_new_profile': True}

Reply:
Got it! I've created a new profile for Wife — I now know they likes dark chocolate, orchids and budget around LKR 5,000. I'll use this to personalise every gift recommendation. Would you like me to find something for them now?


## Test 3 — Search again (now personalised with saved profile)

In [9]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Find an anniversary gift for my wife',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 16:44:03.522 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 11:14:04 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 16:44:04.008 | INFO     | agents.router:classify:125 - Intent: search (98%) — Asking for gift recommendations for a specific

Intent     : search (98%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Find an anniversary gift for my wife | preferences: dark chocolate, orchids | budget LKR 5000', 'recipient': 'Wife'}

Reply:
I'd be happy to help you find a special anniversary gift for your wife. 

Considering your wife's love for orchids, I think the 'Adaraniya Amma' Luxe Orchid Ribbon Mother'S Day Cake (https://www.kapruka.com/buyonline/adaraniya-amma-luxe-orchid-rib/kid/cake00ka002153) would be a stunning choice, despite being slightly above your budget of LKR 5000 at LKR 7,800. This beautiful cake features orchids, which aligns perfectly with her interests. However, I understand that it might be a bit pricey.

A more budget-friendly option would be the Revello Speciality Nougat 100g (https://www.kapruka.com/buyonline/revello-speciality-nougat-100g/kid/chocolates001172) priced at LKR 800, which falls within your budget and caters to her love for dark chocolate (although it's not ex

## Test 4 — Logistics check

In [10]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Can you deliver to Jaffna by this weekend?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 16:44:44.170 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 11:14:44 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 16:44:44.666 | INFO     | agents.router:classify:125 - Intent: logistics_check (98%) — Asking about delivery feasibility to 

Intent     : logistics_check (98%)
Session    : s001
Metadata   : {'district': 'Jaffna', 'feasible': True}

Reply:
I'm happy to help you with delivery to Jaffna. Yes, we do deliver to Jaffna, and the estimated delivery time is 3 days. Please note that Jaffna falls under our extended delivery zone, which takes 3-5 days to reach. 

There is a delivery surcharge of LKR 600 applicable to this region. Since it's an extended zone, I would advise you to order well in advance, especially if you're planning for a special occasion with a fixed date.

Unfortunately, it's unlikely that we can deliver by this weekend, given the 3-day delivery timeframe. However, if you place your order now, we can ensure that it reaches Jaffna as soon as possible.

Now that we've confirmed delivery, would you like me to help you find a suitable gift? I'd be more than happy to assist you in choosing something special for your loved ones in Jaffna.


## Test 5 — Logistics check (same-day zone)

In [11]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Do you deliver to Colombo today?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 16:45:07.225 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 11:15:07 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 16:45:07.625 | INFO     | agents.router:classify:125 - Intent: logistics_check (99%) — Asking about same-day delivery to a s

Intent     : logistics_check (99%)
Session    : s001
Metadata   : {'district': 'Colombo', 'feasible': True}

Reply:
Yes, we do deliver to Colombo, and the best part is that it's a same-day delivery zone. If you place your order before 11 AM, we can get your items to you today. There's no additional surcharge for delivery to Colombo, so you won't have to worry about any extra costs. Now that we've confirmed delivery, would you like some help finding a perfect gift? I'm here to assist you in making your occasion even more special.


## Test 6 — Chitchat / greeting

In [ ]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Hi! What can you help me with?',
)
show(r)

## Test 7 — New recipient profile

In [ ]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='My dad is turning 60, he loves cricket and books, budget LKR 8000',
)
show(r)

## Test 8 — Search for new recipient

In [ ]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='What should I get dad for his birthday?',
)
show(r)

## View conversation history

In [12]:
from memory.st_store import SupabaseSTStore

st = SupabaseSTStore()
history = st.recent('u001', 's001', k=20)

for turn in history:
    role = turn.role.upper()
    print(f'[{role}] {turn.content[:120]}')
    print()

[USER] Find a birthday gift for my wife who loves chocolate

[ASSISTANT] I searched the kapruka.com catalog but couldn't find products that match your request closely enough. Could you give me 

[USER] Find a birthday gift for my wife who loves chocolate

[ASSISTANT] I searched the kapruka.com catalog but couldn't find products that match your request closely enough. Could you give me 

[USER] Find a birthday gift for my wife who loves chocolate

[ASSISTANT] I searched the kapruka.com catalog but couldn't find products that match your request closely enough. Could you give me 

[USER] Find a birthday gift for my wife who loves chocolate

[ASSISTANT] I searched the kapruka.com catalog but couldn't find products that match your request closely enough. Could you give me 

[USER] Find a birthday gift for my wife who loves chocolate

[ASSISTANT] I'd be delighted to help you find a wonderful birthday gift for your wife. Since she loves chocolate, I've shortlisted s

[USER] Find a dinasoree t

## View saved recipient profiles

In [13]:
from memory.profile_store import SupabaseProfileStore

ps = SupabaseProfileStore()
profiles = ps.list_profiles('u001')

for p in profiles:
    print(f'Name        : {p.name}')
    print(f'Relationship: {p.relationship}')
    print(f'Preferences : {p.preferences}')
    print(f'Dislikes    : {p.dislikes}')
    print(f'Budget LKR  : {p.budget_lkr}')
    print()

Name        : Wife
Relationship: wife
Preferences : ['dark chocolate', 'orchids']
Dislikes    : []
Budget LKR  : 5000

